# GNN ile Max-Cut: etiketsiz (unsupervised) küçük bir örnek

Bu notebook, bir GNN'nin doğrudan 'optimal çözümü ezberlemesi' yerine, graph yapısını kullanarak bir ayrık optimizasyon problemi için **diferansiyellenebilir bir surrogate objective** öğrenmesini gösterir.

Amaç: $G=(V,E)$ grafında düğümleri iki kümeye ayırıp kümeler arasından geçen kenar sayısını maksimize etmek.

GNN her düğüm için $p_i\in(0,1)$ üretir. Bir kenarın kesitte olmasının yumuşak karşılığı
$$p_i(1-p_j)+(1-p_i)p_j$$
olarak alınır. Bunun negatifini minimize ederiz; eğitim sonunda $p_i\ge 0.5$ ile ayrık çözüme yuvarlarız.

In [ ]:
import random
import numpy as np
import networkx as nx
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Problem örneğini oluştur

Küçük bir Erdős–Rényi grafı kullanıyoruz. Simetrileri kırmak için her düğüme sabit seed ile üretilmiş rastgele başlangıç özellikleri veriyoruz. Bu tek-instance eğitim örneğinde pedagojik bir seçimdir; çok-instance inductive eğitimde problemden gelen gerçek node feature'ları ve uygun positional/structural encoding'ler tercih edilmelidir.

In [ ]:
n = 18
edge_probability = 0.28
G = nx.erdos_renyi_graph(n=n, p=edge_probability, seed=SEED)

edges = list(G.edges())
edge_index = torch.tensor(
    [(u, v) for u, v in edges] + [(v, u) for u, v in edges],
    dtype=torch.long,
).t().contiguous()

# Tek graph üzerinde optimizasyon yaparken simetriyi kıran başlangıç özellikleri.
x = torch.randn((n, 8), dtype=torch.float32)
data = Data(x=x, edge_index=edge_index)
print(f'Düğüm: {G.number_of_nodes()}, kenar: {G.number_of_edges()}')

## 2. Basit GCN modeli

Bu örnek bilinçli olarak küçük tutulmuştur. Gerçek OR uygulamalarında GAT, GIN, GraphSAGE, heterojen GNN veya Graph Transformer gibi modeller problem yapısına göre daha uygun olabilir.

In [ ]:
class MaxCutGCN(nn.Module):
    def __init__(self, in_dim=8, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        h = torch.relu(self.conv1(data.x, data.edge_index))
        h = torch.relu(self.conv2(h, data.edge_index))
        return torch.sigmoid(self.head(h)).squeeze(-1)

model = MaxCutGCN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

## 3. Diferansiyellenebilir Max-Cut amacı

PyG `edge_index` yönlü biçimde tutulduğu için her orijinal kenarı bir kez hesaba katmak amacıyla NetworkX kenar listesini kullanıyoruz.

In [ ]:
u = torch.tensor([a for a, b in edges], dtype=torch.long)
v = torch.tensor([b for a, b in edges], dtype=torch.long)

def soft_cut_value(p):
    return (p[u] * (1 - p[v]) + (1 - p[u]) * p[v]).sum()

history = []
for epoch in range(600):
    optimizer.zero_grad()
    p = model(data)
    loss = -soft_cut_value(p)
    loss.backward()
    optimizer.step()
    history.append(-loss.item())

print(f'Son soft cut değeri: {history[-1]:.3f}')

## 4. Yuvarlama ve değerlendirme

Sinir ağının çıktısını ayrık çözüme çeviriyoruz. Küçük örnekte gerçek optimumu brute force ile de hesaplayarak optimality gap görüyoruz. Bu brute-force bölümü yalnızca eğitim örneğini doğrulamak içindir.

In [ ]:
with torch.no_grad():
    probabilities = model(data)
assignment = (probabilities >= 0.5).long().numpy()

def cut_value(bits):
    return sum(int(bits[a] != bits[b]) for a, b in edges)

gnn_cut = cut_value(assignment)

best_cut = -1
best_bits = None
# Simetriyi kırmak için ilk düğümü 0'a sabitliyoruz.
for mask in range(1 << (n - 1)):
    bits = [0] + [(mask >> i) & 1 for i in range(n - 1)]
    value = cut_value(bits)
    if value > best_cut:
        best_cut = value
        best_bits = bits

gap = (best_cut - gnn_cut) / best_cut if best_cut else 0.0
print('GNN cut:', gnn_cut)
print('Optimal cut:', best_cut)
print(f'Optimality gap: %{100*gap:.2f}')

## 5. Bu örnek nasıl geliştirilir?

- Aynı dağılımdan çok sayıda graph instance üretip tek bir modeli **inductive** olarak eğitin.
- GCN yerine GraphSAGE, GIN, GAT veya heterophily-aware modelleri karşılaştırın.
- Tek eşik yerine randomized rounding + local search kullanın.
- Sonucu yalnız objective ile değil; çözüm süresi, optimality gap, farklı graph büyüklüklerine genelleme ve farklı graph dağılımlarına dayanıklılık ile ölçün.
- GNN'yi tek başına solver yapmak yerine, GNN skorlarını klasik local search / MIP / CP-SAT solver'a başlangıç veya arama yönlendirmesi olarak verin.

Ana rehber için repo kökündeki `README.md` dosyasına bakın.